In [ ]:
# ruff: noqa
import argparse
import logging
import re
import sys
import warnings
from pathlib import Path
import cellseg_benchmark  # noqa: F401
import anndata as ad
import anndata2ri
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc
from rpy2.rinterface_lib.embedded import RRuntimeError
from rpy2.robjects.conversion import localconverter

plt.rcParams["font.family"] = (
    "Arial" if "Arial" in [f.name for f in fm.fontManager.ttflist] else "sans-serif"
)
plt.rcParams["font.weight"] = "normal"

sys.path.insert(1, "/Users/sebert/Downloads/cellseg-benchmark")
from cellseg_benchmark._constants import cell_type_colors
from cellseg_benchmark.adata_utils import plot_pseudobulk_pca
from cellseg_benchmark.dea_utils import (
    add_ensembl_id,
    add_group_sample_counts,
    prepare_adata_for_rpy2,
    pseudobulk_aggregate_and_filter,
    safe_sheet,
)

warnings.filterwarnings("ignore", message=".*Observation names are not unique*")
VALID_METHODS = {"LRT", "QL"}


def get_args(test_args=None):  # noqa: D103
    p = argparse.ArgumentParser(description="Run edgeR pseudobulk-based DEA")
    p.add_argument("cohort", help="Cohort name, e.g. foxf2")
    p.add_argument(
        "seg_method", help="Segmentation method, e.g. Cellpose_1_nuclei_model"
    )
    p.add_argument(
        "--sample_key", default="sample", help="obs column for donor/sample ID"
    )
    p.add_argument(
        "--subset_key",
        default="cell_type",
        help="obs column used to subset data (e.g. cell_type, cluster, region)",
    )
    p.add_argument(
        "--subset_values",
        nargs="+",
        default=None,
        help="Values of subset_key to process (default: all unique values)",
    )
    p.add_argument(
        "--condition_key",
        default="genotype",
        help="obs column for condition (e.g. genotype)",
    )
    p.add_argument(
        "--batch_key",
        default="slide",
        help="optional batch key for inclusion as covariate (default: slide)",
    )
    p.add_argument("--ref", default="WT", help="Reference group (default: WT)")
    p.add_argument(
        "--test_groups",
        nargs="+",
        default=None,
        help="Groups to test vs reference (default: all groups in condition_key except --ref)",
    )
    p.add_argument(
        "--edger_methods",
        nargs="+",
        default=["LRT"],
        choices=sorted(VALID_METHODS),
        help="edgeR method(s) to run (default: LRT)",
    )
    p.add_argument("--min_cells", type=int, default=15, help="Minimum cells per donor")
    p.add_argument(
        "--replicates_per_patient",
        type=int,
        default=3,
        help="Number of pseudoreplicates per donor",
    )
    p.add_argument(
        "--overwrite",
        type=lambda x: str(x).lower() in ["true", "1", "yes"],
        default=True,
        help="Overwrite existing result files (default: True)",
    )
    if test_args is not None:
        return p.parse_args(test_args)
    else:
        return p.parse_args()

In [ ]:
# Simulate CLI arguments inside Jupyter
args = get_args(
    [
        "foxf2",
        "Cellpose_1_nuclei_model",
        "--sample_key",
        "batch",
        "--subset_key",
        "cell_type",
        "--condition_key",
        "batch",
        "--ref",
        "HFD",
        "--edger_methods",
        "LRT",
        "--overwrite",
        "False",
    ]
)

In [ ]:
# Simulate CLI arguments inside Jupyter
args = get_args(
    [
        "aging",
        "Cellpose_1_nuclei_model",
        "--sample_key",
        "batch",
        "--subset_key",
        "cell_type",
        "--condition_key",
        "batch",
        "--ref",
        "HFD",
        "--edger_methods",
        "LRT",
        "--overwrite",
        "False",
    ]
)

In [ ]:
args

In [ ]:
# Logger setup
logger = logging.getLogger("dea")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s]: %(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    logger.propagate = False
rcb.logger.handlers = logger.handlers
#rcb.consolewrite_print = lambda x: logger.debug(f"R: {x.strip()}")
rcb.consolewrite_error = lambda x: (_ for _ in ()).throw(RRuntimeError(x.strip()))
#rcb.consolewrite_message = lambda x: logger.info(f"R: {x.strip()}")
rcb.consolewrite_warn = lambda x: (
    logger.warning if x.lstrip().lower().startswith("warning") else logger.info
)(f"R: {x.strip()}")
setattr(rcb, "consolewrite_warnerror", rcb.consolewrite_warn)

In [ ]:
# R setup
conv = ro.default_converter + ro.pandas2ri.converter + anndata2ri.converter
r_script = Path(sys.path[1]) / "cellseg_benchmark" / "dea_utils.r"
ro.r["source"](str(r_script))
edgeR_loop = ro.globalenv["edgeR_loop"]

In [ ]:
base_path = Path("/Users/sebert/Downloads/cellseg-benchmark")
method_path = base_path / "analysis" / args.cohort / args.seg_method
output_dir = method_path / "dea"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
logger.info("Loading integrated AnnData...")
adata = sc.read_h5ad("/Volumes/Simon Ebert/snRNAseq_Python_env_normalMT/integrated_harmony.h5ad")


In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
# re-group cell types
adata.obs["cell_type"] = (
    adata.obs["cell_type"]
    .astype(str)

    .astype("category")
)

In [ ]:
adata.obs["cell_type"].value_counts()

In [ ]:
# Clean up group names for R conversion
adata.obs[args.subset_key] = [
    key.replace(" ", "_")
    .replace("-", "_")
    .replace("/", "_")
    .replace("+", "")
    .replace("(", "")
    .replace(")", "")
    if not pd.isna(key)
    else key
    for key in adata.obs[args.subset_key]
]
for col in [args.condition_key, args.sample_key, args.subset_key]:
    adata.obs[col] = adata.obs[col].astype("category")
if args.test_groups is None:
    groups = adata.obs[args.condition_key].dropna().unique().tolist()
    args.test_groups = [g for g in groups if g != args.ref]
    msg = (
        f"Test groups inferred from '{args.condition_key}': {', '.join(map(str, args.test_groups))}"
        if args.test_groups
        else f"No test groups found different from ref '{args.ref}' under '{args.condition_key}'."
    )
    logger.info(msg)

# print sample overview
if args.subset_values is None:
    groups_to_process = sorted(adata.obs[args.subset_key].unique())
else:
    groups_to_process = args.subset_values
for cond, df in adata.obs.groupby(args.condition_key, observed=True):
    samples = df[args.sample_key].unique()
    logger.info(f"{cond}: {len(samples)} samples → {', '.join(samples)}")

In [ ]:
adata.obs["n_cells_sum"] = adata.obs.groupby(
    [args.subset_key, args.sample_key], observed=True
)[args.sample_key].transform("count")


In [ ]:
adata

In [ ]:
adata.X = adata.layers["soupX_counts"].copy().astype(int)
assert np.issubdtype(adata.X.dtype, np.integer)

In [ ]:
obs_to_keep = [
    args.condition_key,
    args.subset_key,
    args.sample_key,
    "n_cells_sum",
   
]
if args.batch_key and args.batch_key in adata.obs.columns:
    obs_to_keep.append(args.batch_key)
else:
    logger.warning(
        f"Batch column '{args.batch_key}' was not found in adata.obs "
        "or is invalid. Continuing **without** a batch covariate."
    )
    args.batch_key = None

In [ ]:
args.replicates_per_patient

In [ ]:
logger.info("Run pseudobulking...")
total = len(groups_to_process)
logger.info(f"Processing {groups_to_process[0]} ({1}/{total})")
adata_pb = pseudobulk_aggregate_and_filter(
    adata,
    subset_value=groups_to_process[0],
    sample_key=args.sample_key,
    subset_key=args.subset_key,
    obs_to_keep=obs_to_keep,
    min_cells=args.min_cells,
    replicates_per_patient=args.replicates_per_patient,
    logger=logger,
)
for i, group in enumerate(groups_to_process[1:], 2):
    logger.info(f"Processing {group} ({i}/{total})")
    adata_pb_i = pseudobulk_aggregate_and_filter(
        adata,
        subset_value=group,
        sample_key=args.sample_key,
        subset_key=args.subset_key,
        obs_to_keep=obs_to_keep,
        min_cells=args.min_cells,
        replicates_per_patient=args.replicates_per_patient,
        logger=logger,
    )
    adata_pb = ad.concat(
        [adata_pb, adata_pb_i], join="outer", label=None, index_unique=None
    )
adata_pb.obs_names_make_unique()
# Add whole brain: pseudobulk per sample across all groups
adata_pb_i = pseudobulk_aggregate_and_filter(
    adata,
    subset_value=None,
    sample_key=args.sample_key,
    subset_key=args.subset_key,
    obs_to_keep=obs_to_keep,
    min_cells=args.min_cells,
    replicates_per_patient=args.replicates_per_patient,
    logger=logger,
)
adata_pb_i.obs[args.subset_key] = "all"
adata_pb = ad.concat([adata_pb, adata_pb_i])
del adata

In [ ]:
adata_pb.obs["volume_sum"] = adata_pb.X.sum(axis=1).A1 if hasattr(adata_pb.X, "A1") else adata_pb.X.sum(axis=1)


In [ ]:
#plot_pseudobulk_pca(adata_pb, args, output_dir, cell_type_colors, logger)

In [ ]:
logger.info("Run DEA...")
adata_pb = prepare_adata_for_rpy2(adata_pb, key=args.subset_key)
adatas_pb = {}
unique_groups = adata_pb.obs[args.subset_key].unique().tolist()
for key in unique_groups:
    tmp = adata_pb[adata_pb.obs[args.subset_key] == key].copy()
    adatas_pb[key] = tmp

In [ ]:
adatas_pb

In [ ]:
print({
    'edger_methods': args.edger_methods,
    'test_groups': args.test_groups,
    'ref_group': args.ref,
    'condition_col': args.condition_key,
    'batch_col': args.batch_key,
})


In [ ]:
# loop over cell types

In [ ]:
adata_pb.obs["n_cells_sum"]

In [ ]:
all_degs = {}
for group_i in adatas_pb:
    adata_tmp = adatas_pb[group_i]
    with localconverter(conv):
        combined_results = edgeR_loop(
            adata=adata_tmp,
            group_i=group_i,
            edger_methods=args.edger_methods,
            test_groups=args.test_groups,
            ref_group=args.ref,
            condition_col=args.condition_key,
            #batch_col=args.batch_key,
        )
    if combined_results is not ro.NULL:
        all_degs[group_i] = combined_results

In [ ]:
# format and export

In [ ]:
# sort each dataframe in all_degs by logFC ascending (in-place replacement)
for k, df in list(all_degs.items()):
    if isinstance(df, pd.DataFrame) and "logFC" in df.columns:
        all_degs[k] = df.sort_values("logFC", ascending=True).reset_index(drop=True)

In [ ]:
all_degs["Ms"]

In [ ]:
logger.info("Format output table...")
collapsed_df = pd.concat(
    [pd.DataFrame(v).assign(subset=k) for k, v in all_degs.items()], ignore_index=True
).drop(columns=["result_id"], errors="ignore")

collapsed_df = add_group_sample_counts(
    collapsed_df,
    adatas_pb,
    condition_key=args.condition_key,
    sample_key=args.sample_key,
    ref=args.ref,
    test_groups=args.test_groups,
    subset_group="subset",
)
collapsed_df = collapsed_df.set_index("gene")
collapsed_df.insert(collapsed_df.columns.get_loc("FC"), "gene", collapsed_df.index)

order = [
    "subset",
    "gene",
    "FC",
    "logFC",
    "PValue",
    "FDR",
    "logCPM",
    "LR",
    "method",
    "test_group",
    "ref",
    "test",
]
collapsed_df = collapsed_df[order + [c for c in collapsed_df.columns if c not in order]]



In [ ]:
collapsed_df

In [ ]:
# keep only significant genes (FDR < 0.05)
collapsed_df = collapsed_df[collapsed_df["FDR"] < 0.05].copy()
collapsed_df

In [ ]:
sc.pl.violin(adata_pb, keys="Stat1", groupby="batch", use_raw=False)

In [ ]:
logger.info("Export table(s)...")
subset_key_clean = re.sub(r"[_-]", "", args.subset_key)

In [ ]:
name = f"{args.cohort}-by-{subset_key_clean}_pseudobulk_edgeR_Ldlr_Leiden"
name

In [ ]:
#name = f"{args.cohort}-by-{subset_key_clean}_pseudobulk_edgeR_libnorm"
#name

In [ ]:
for (method, test), df_mt in collapsed_df.groupby(["method", "test"]):
    df_mt = df_mt.loc[:, df_mt.notna().any()]  # drop all-NaN cols
    xlsx = output_dir / f"{name}_{method.split("_", 1)[-1]}_{test}.xlsx"
    if xlsx.exists() and not args.overwrite:
        logger.info(f"  Exists, skip: {xlsx}")
        continue
    used = set()
    with pd.ExcelWriter(xlsx, engine="xlsxwriter") as writer:
        for gid, g in df_mt.groupby("subset", sort=True):
            g.sort_values("PValue").to_excel(
                writer, sheet_name=safe_sheet(gid, used), index=False
            )
    logger.info(f"  Wrote: {xlsx}")

logger.info("Done.")

In [ ]:
# Calculate the intersect between the methods

In [ ]:
xlsx_path = Path("/Volumes/Simon Ebert/snRNAseq_Python_env_normalMT/DEG_List_new/All_celltypes_DEG.xlsx")
if not xlsx_path.exists():
    logger.error(f"File not found: {xlsx_path}")
    raise FileNotFoundError(xlsx_path)

degs_MAST = pd.read_excel(xlsx_path, sheet_name="Ms")
logger.info(f"Loaded sheet 'B' from {xlsx_path} — shape: {degs_MAST.shape}")
degs_MAST

In [ ]:
xlsx_path = Path("/Users/sebert/Downloads/cellseg-benchmark/analysis/aging/Cellpose_1_nuclei_model/dea/aging-by-celltype_pseudobulk_edgeR_Ldlr_Leiden_LRT_HFD_L1vsHFD.xlsx")

if not xlsx_path.exists():
    logger.error(f"File not found: {xlsx_path}")
    raise FileNotFoundError(xlsx_path)

degs_edgeR = pd.read_excel(xlsx_path, sheet_name="Ms")
logger.info(f"Loaded sheet 'B' from {xlsx_path} — shape: {degs_edgeR.shape}")
degs_edgeR

In [ ]:
degs_MAST

In [ ]:
df_ms=degs_edgeR

In [ ]:
df_ms.head()

In [ ]:
unwanted_patterns = ['Rik$', '^Gm', 'Rpl', 'Rps', 'Mrpl', 'Mrps', 'mt-']
filter_regex = '|'.join(unwanted_patterns)
df_ms = df_ms[~df_ms['gene'].str.contains(filter_regex, regex=True)]


In [ ]:
# keep only significant upregulated genes
df_ms = df_ms.loc[(df_ms["FDR"] < 0.05) & (df_ms["logFC"] > 0.5)].copy()
print(f"Filtered shape: {df_ms.shape}")
df_ms.head()

In [ ]:
degs_BC = degs_MAST.loc[(degs_MAST["adj_pVal"] < 0.01) & (degs_MAST["logFC"] > 0.5)].copy()
print(f"Filtered shape: {degs_BC.shape}")
degs_BC.head()

In [ ]:
# intersect genes between df['gene'] and degs_BC['genes']
genes_df = set(df_ms['gene'].astype(str))
genes_degs = set(degs_BC['gene_symbol'].astype(str))
intersect_genes = sorted(genes_df & genes_degs)

print(f"Intersect count: {len(intersect_genes)}")
print("First 50 intersecting genes:", intersect_genes[:50])

# expose as dataframe if needed
intersect_df = pd.DataFrame({"gene": intersect_genes})
intersect_df.head()

In [ ]:
#Intersect over all genes

In [ ]:
from pathlib import Path
import pandas as pd

# Paths
mast_path = Path("/Volumes/Simon Ebert/snRNAseq_Python_env_normalMT/DEG_List_new/All_celltypes_DEG.xlsx")
edger_path = Path("/Users/sebert/Downloads/cellseg-benchmark/analysis/aging/Cellpose_1_nuclei_model/dea/aging-by-celltype_pseudobulk_edgeR_Ldlr_Leiden_LRT_HFD_L1vsHFD.xlsx")

if not mast_path.exists():
    raise FileNotFoundError(mast_path)
if not edger_path.exists():
    raise FileNotFoundError(edger_path)

# Load
mast_sheets = pd.read_excel(mast_path, sheet_name=None)
edger_sheets = pd.read_excel(edger_path, sheet_name=None)

# Match sheets
common_sheets = set(mast_sheets.keys()) & set(edger_sheets.keys())
print(f"Common cell types: {common_sheets}")

# Gene filtering
unwanted_patterns = ['Rik$', '^Gm', 'Rpl', 'Rps', 'Mrpl', 'Mrps', 'mt-']
filter_regex = '|'.join(unwanted_patterns)

summary = []
results = {}

for celltype in common_sheets:
    print(f"\nProcessing: {celltype}")

    mast_df = mast_sheets[celltype].copy()
    edger_df = edger_sheets[celltype].copy()

    # --- Column handling (robust) ---
    mast_gene_col = 'gene_symbol' if 'gene_symbol' in mast_df.columns else 'gene'
    edger_gene_col = 'gene_symbol' if 'gene_symbol' in edger_df.columns else 'gene'

    # --- Filter MAST ---
    mast_df = mast_df[~mast_df[mast_gene_col].astype(str).str.contains(filter_regex, regex=True, na=False)]
    mast_df = mast_df[
        (mast_df["adj_pVal"] < 0.01) & (mast_df["logFC"] > 0.5)
    ]

    # --- Filter edgeR ---
    edger_df = edger_df[~edger_df[edger_gene_col].astype(str).str.contains(filter_regex, regex=True, na=False)]
    edger_df = edger_df[
        (edger_df["FDR"] < 0.05) & (edger_df["logFC"] > 0.5)
    ]

    # Gene sets
    mast_genes = set(mast_df[mast_gene_col].astype(str))
    edger_genes = set(edger_df[edger_gene_col].astype(str))

    # Intersection
    intersect = sorted(mast_genes & edger_genes)

    # Sizes
    n_mast = len(mast_genes)
    n_edger = len(edger_genes)
    n_inter = len(intersect)
    n_union = len(mast_genes | edger_genes)

    # Percentages
    pct_mast = (n_inter / n_mast * 100) if n_mast else 0
    pct_edger = (n_inter / n_edger * 100) if n_edger else 0
    pct_jaccard = (n_inter / n_union * 100) if n_union else 0

    print(f"Intersect: {n_inter} | %MAST: {pct_mast:.2f}% | %edgeR: {pct_edger:.2f}% | Jaccard: {pct_jaccard:.2f}%")

    # Store intersect genes
    results[celltype] = pd.DataFrame({"gene": intersect})

    # Store summary
    summary.append({
        "celltype": celltype,
        "n_MAST": n_mast,
        "n_edgeR": n_edger,
        "n_intersect": n_inter,
        "pct_MAST": pct_mast,
        "pct_edgeR": pct_edger,
        "pct_jaccard": pct_jaccard
    })

# Final table
summary_df = pd.DataFrame(summary).sort_values("pct_jaccard", ascending=False)

print("\n=== FINAL SUMMARY ===")
print(summary_df)
summary_df.to_excel("/Users/sebert/Downloads/cellseg-benchmark/analysis/aging/Cellpose_1_nuclei_model/dea/edgeR_vs_Mast.xlsx")